In [1]:
import time

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

from nn import MLP
from tensor import Tensor

In [2]:
mnist = fetch_openml(
    "mnist_784",
    version=1,
    as_frame=False
)

X = mnist.data
y = mnist.target.astype(int)

In [3]:
n_samples = 50000
lr = 0.05

model = MLP(784, [16, 16, 10])

X_train = Tensor(X[:n_samples] / 255.0)
y_train = y[:n_samples]

In [4]:
def cross_entropy(probs, target):
    n = len(target)

    one_hot = np.zeros_like(probs.data)
    one_hot[np.arange(n), target] = 1.0
    
    loss = (-probs.log() * Tensor(one_hot)).sum() * (1 / n)
    return loss

def loss_fn():
    logits = model(X_train)
    probs = logits.softmax()
    return cross_entropy(probs, y_train)

def loss_fn_optimized():
    logits = model(X_train)
    return logits.cross_entropy_softmax(y_train)

start_time = time.perf_counter()
for epoch in range(1000):
    model.zero_grad()
    loss = loss_fn_optimized()
    # loss = loss_fn()
    
    loss.backward()

    for p in model.parameters():
        p.data -= lr * p.grad

    if epoch % 50 == 0:
        print(f"epoch {epoch} loss {loss.data}")

elapsed_time = time.perf_counter() - start_time
print(f"\nTotal time: {elapsed_time:.4f} seconds")
print(f"Average time per epoch: {elapsed_time / 1000:.6f} seconds")

epoch 0 loss 2.339250087738037
epoch 50 loss 1.2958004474639893
epoch 100 loss 0.8084878921508789
epoch 150 loss 0.6178378462791443
epoch 200 loss 0.5173066258430481
epoch 250 loss 0.45692116022109985
epoch 300 loss 0.4167957007884979
epoch 350 loss 0.3880659341812134
epoch 400 loss 0.3663759231567383
epoch 450 loss 0.3492051661014557
epoch 500 loss 0.3352183699607849
epoch 550 loss 0.3235173523426056
epoch 600 loss 0.3135508596897125
epoch 650 loss 0.30489382147789
epoch 700 loss 0.2972946763038635
epoch 750 loss 0.29053395986557007
epoch 800 loss 0.2844691276550293
epoch 850 loss 0.27895936369895935
epoch 900 loss 0.27391794323921204
epoch 950 loss 0.2692883312702179

Total time: 110.4825 seconds
Average time per epoch: 0.110483 seconds


In [5]:
test_samples = 10000
X_test = Tensor(X[n_samples:n_samples + test_samples] / 255.0)
y_test  = y[n_samples:n_samples + test_samples]

logits = model(X_test)
probs  = logits.softmax()
y_pred = np.argmax(probs.data, axis=1)

correct = (y_pred == y_test).sum()
print(f"\ntest accuracy: {correct}/{test_samples}  ({100 * correct / test_samples}%)")


test accuracy: 9304/10000  (93.04%)
